# 22 Risks and rates on real data: R's `poisson.test` and `mantelhaen.test` examples, UCB admissions, lung

`table.compare_rates`, `sdata.compare_rates`, `table.rate_of`, `table.compare_risks`, `sdata.compare_risks` and
`table.compare_balanced_groups_of` (`against=`, `interval=`), with every option the gap list names.

**Only simplydata runs in this notebook.** The reference values are computed by
`scripts/references/22_risks_and_rates.py` and stored in `references/22_risks_and_rates.csv`, each with its
source, its kind (printed, certified, or a run of the reference package) and a link to the manual page: R
`poisson.test`, `prop.test`, `fisher.test`, `binom.test`, `mantelhaen.test` and DescTools `BreslowDayTest`
(simplydata calls statsmodels and SciPy for these), and formulas written out (Garwood's exact Poisson interval,
the conditional binomial test and interval of a rate ratio, Newcombe's hybrid score interval, Wald intervals on
the log scale, the Mantel-Haenszel risk ratio with Greenland & Robins' variance, the two one-sided tests).

Data: the second example of R's `poisson.test` help page (lung cancer in Fredericia against three other Danish
cities, Andersen 1977), the NCCTG lung cancer trial (R survival), the UC Berkeley admissions of 1973 (Bickel,
Hammel & O'Connell 1975; the example of R's `mantelhaen.test`), and lalonde (stored in `data/`).

Tolerances (stored with each reference): closed formulas 1e-10 relative; the propensity fits 1e-6.

In [1]:
import simplydata as sdata

from checks import ReferenceChecks, data_file, quiet, read_row, read_rows, versions

NOTEBOOK = "22_risks_and_rates"
versions("simplydata")
checks = ReferenceChecks(NOTEBOOK)


def load(name):
    return quiet(sdata.table, data_file(NOTEBOOK, name), separator=",")


def rows(word, *args, **options):
    return read_rows(quiet(word, *args, **options))


def rate_values(found):
    """The numbers of a compare_rates answer, named as the references name them."""
    values = {}
    for row in found:
        if row["measure"] == "rate":
            for column in row:
                if column == "rate":
                    values[f"rate, {row['group']}"] = row[column]
                elif column.startswith("rate ") and "CI" in column:
                    values[f"{column}, {row['group']}"] = row[column]
        elif row["measure"].startswith("rate ratio"):
            for column in row:
                if column == "estimate":
                    values["rate ratio"] = row[column]
                elif column.startswith("estimate ") and "CI" in column:
                    values["rate ratio" + column[len("estimate"):]] = row[column]
                elif column.startswith("p-value"):
                    values[column] = row[column]
    return values

Python 3.12.16 on Linux 7.0.0-1020-azure
simplydata 0.2.0 (commit a5a8ff0 of github.com/bioforkgithub/simplydata)
simplydata 0.2.0


R version 4.6.2 (2026-10-29)


## Rates: Fredericia against three other cities (R `poisson.test` help page)

The counts are read from the cities table with simplydata (`get_cell_at`), for `sdata.compare_rates`.

In [2]:
towns = load("cities")
towns.compare_rates("cases", time="person_years", for_each_group_in="city", against="other cities")

"cases" per 1000 "person_years": 13.8 in Fredericia (95 % CI 6.86 to 24.6; 11 events in 800) and 6.97 in other cities (95 % CI 4.32 to 10.7; 21 events in 3011).


Rate ratio 1.97 (95 % CI 0.858 to 4.28): the rate in Fredericia is 1.97 times the rate in other cities.
If the rates were truly the same, a difference this big or bigger would turn up in about 8 of 100 studies like this one (p = 0.080). This does not show that the rates are the same.
Method: exact intervals for each rate (statsmodels confint_poisson, as R poisson.test()); the exact conditional test of the rate ratio (statsmodels test_poisson_2indep), with its exact interval from R, as R poisson.test(c(11, 21), c(800, 3011)).


,group,events,time,rate,rate 95 % CI low,rate 95 % CI high,per,measure,against,estimate,estimate 95 % CI low,estimate 95 % CI high,p-value,method
1,Fredericia,11.0,800.0,13.750000,6.86395,24.602548,1000 person_years,rate,None,NaN,NaN,NaN,NaN,None
2,other cities,21.0,3011.0,6.974427,4.31728,10.661153,1000 person_years,rate,None,NaN,NaN,NaN,NaN,None
3,Fredericia,NaN,NaN,NaN,NaN,NaN,None,rate ratio,other cities,1.971488,0.858426,4.277266,0.079669,exact conditional test (statsmodels test_poiss...


In [3]:
ds = "lung cancer, Fredericia and other cities (R poisson.test example; Andersen 1977)"
cases = [towns.get_cell_at(row=i, column="cases") for i in (1, 2)]
years = [towns.get_cell_at(row=i, column="person_years") for i in (1, 2)]
cities = [towns.get_cell_at(row=i, column="city") for i in (1, 2)]
calls = {"table.compare_rates": lambda **o: rows(towns.compare_rates, "cases", time="person_years", for_each_group_in="city",
                                                  against="other cities", **o),
         "sdata.compare_rates": lambda **o: rows(sdata.compare_rates, events=cases, time=years, names=cities,
                                                  against="other cities", **o)}
for word, call in calls.items():
    for per, interval in [(1000, 95), (100000, 90)]:
        checks.check_values(ds, f"{word} (per={per}, interval={interval})", rate_values(call(per=per, interval=interval)))
    for options in [{"sides": 1, "higher": "Fredericia"}, {"sides": 1, "lower": "Fredericia"}, {"alternative": "greater"}]:
        name = f"{word} (" + ", ".join(f"{k}={v!r}" for k, v in options.items()) + ")"
        checks.check(ds, name, "p-value", rate_values(call(**options))["p-value"])
    checks.check_values(ds, f"{word} (same_within=)", rate_values(call(same_within=[0.5, 2])))
    for option in ("not_lower", "not_higher"):
        checks.check(ds, f'{word} (same_within=, {option}="Fredericia")', "p-value",
                     rate_values(call(same_within=[0.5, 2], **{option: "Fredericia"}))["p-value"])

agrees  lung cancer, Fredericia and other cities (R poisson.test example; Andersen 1977) | table.compare_rates (per=1000, interval=95) | rate, Fredericia: simplydata 13.75, events / time 13.75; relative difference 0.00e+00 (allowed 1e-10)
      Source: events / time (run reference)
agrees  lung cancer, Fredericia and other cities (R poisson.test example; Andersen 1977) | table.compare_rates (per=1000, interval=95) | rate 95 % CI low, Fredericia: simplydata 6.86395045905, Garwood exact, SciPy 6.86395045905; relative difference 0.00e+00 (allowed 1e-10)
      Source: Garwood exact, SciPy (run reference)
agrees  lung cancer, Fredericia and other cities (R poisson.test example; Andersen 1977) | table.compare_rates (per=1000, interval=95) | rate 95 % CI high, Fredericia: simplydata 24.6025481416, Garwood exact, SciPy 24.6025481416; relative difference 0.00e+00 (allowed 1e-10)
      Source: Garwood exact, SciPy (run reference)
agrees  lung cancer, Fredericia and other cities (R poisson.test e

## `rate_of` and `compare_rates` on rows: deaths per 1000 days by sex in the NCCTG lung cancer trial

In [4]:
trial = load("lung")
ds = "NCCTG lung cancer trial (R survival lung)"
for per, interval in [(1000, 95), (365.25, 99)]:
    values = {}
    for row in rows(trial.rate_of, "died", time="time", for_each_group_in="sex", per=per, interval=interval):
        for column in row:
            if column == "rate" or (column.startswith("rate ") and "CI" in column):
                values[f"{column}, {row['group']}"] = row[column]
    checks.check_values(ds, f"rate_of (per={per}, interval={interval})", values)
ratio = [row for row in rows(trial.compare_rates, "died", time="time", for_each_group_in="sex", against="female")
         if row["measure"].startswith("rate ratio")][0]
checks.check_values(ds, 'compare_rates (against="female")', {f"rate ratio {k}": v for k, v in ratio.items()})

agrees  NCCTG lung cancer trial (R survival lung) | rate_of (per=1000, interval=95) | rate, female: simplydata 1.73730619202, events / time 1.73730619202; relative difference 0.00e+00 (allowed 1e-10)
      Source: events / time (run reference)
agrees  NCCTG lung cancer trial (R survival lung) | rate_of (per=1000, interval=95) | rate 95 % CI low, female: simplydata 1.30136155415, Garwood exact, SciPy 1.30136155415; relative difference 0.00e+00 (allowed 1e-10)
      Source: Garwood exact, SciPy (run reference)
agrees  NCCTG lung cancer trial (R survival lung) | rate_of (per=1000, interval=95) | rate 95 % CI low, female: simplydata 1.30136155415, R poisson.test 1.30136155415; relative difference 0.00e+00 (allowed 1e-10)
      Source: R poisson.test (run reference) - https://search.r-project.org/R/refmans/stats/html/poisson.test.html
agrees  NCCTG lung cancer trial (R survival lung) | rate_of (per=1000, interval=95) | rate 95 % CI high, female: simplydata 2.27243927391, Garwood exact, SciP

agrees  NCCTG lung cancer trial (R survival lung) | compare_rates (against="female") | rate ratio estimate: simplydata 1.64937887329, R poisson.test 1.64937887329; relative difference 0.00e+00 (allowed 1e-10)
      Source: R poisson.test (run reference) - https://search.r-project.org/R/refmans/stats/html/poisson.test.html
agrees  NCCTG lung cancer trial (R survival lung) | compare_rates (against="female") | rate ratio estimate 95 % CI low: simplydata 1.17940335468, R poisson.test 1.17940335468; relative difference 0.00e+00 (allowed 1e-10)
      Source: R poisson.test (run reference) - https://search.r-project.org/R/refmans/stats/html/poisson.test.html
agrees  NCCTG lung cancer trial (R survival lung) | compare_rates (against="female") | rate ratio estimate 95 % CI high: simplydata 2.33223899539, R poisson.test 2.33223899539; relative difference 0.00e+00 (allowed 1e-10)
      Source: R poisson.test (run reference) - https://search.r-project.org/R/refmans/stats/html/poisson.test.html
agr

4

## Risks: UC Berkeley admissions, 1973

Crude (women against men), one-sided, Fisher's exact test, equivalence within 5 percentage points and the two
non-inferiority forms, from the rows (`table.compare_risks`) and from the counts (`sdata.compare_risks`, the counts
found with simplydata's `count_rows`).

In [5]:
berkeley = load("applicants")
berkeley.compare_risks("admit", for_each_group_in="sex", against="Male", value="Admitted")

"admit" = "Admitted" (chosen with value=): 557 of 1835 (30.4 %) in Female and 1198 of 2691 (44.5 %) in Male.
Out of 100, about 14 fewer in Female than in Male (risk difference -14.2 percentage points, 95 % CI -17.0 to -11.3).
Risk ratio 0.682 (95 % CI 0.629 to 0.739): the risk in Female is 0.682 times the risk in Male. Odds ratio 0.543 (95 % CI 0.479 to 0.616); odds are not risks: with an outcome this common (over 10 %) the odds ratio lies further from 1 than the risk ratio.
Number needed for one event fewer: 8 (1 / 0.142, rounded up; 95 % CI 5.90 to 8.83): for every 8 in Female instead of Male, one event fewer (the NNTB, if the event is harmful, such as death).
If the risks were truly the same, a difference this big or bigger would turn up in fewer than 1 in 1000 studies like this one (p < 0.0001).
Method: the chi-square test with Yates' correction (SciPy, as R prop.test()) for the p-value; exact intervals for each risk, Newcombe's score interval for the difference (as R DescTools::Bi

,measure,group,against,estimate,estimate 95 % CI low,estimate 95 % CI high,events,out of,p-value,method
1,risk,Female,None,0.303542,0.282560,0.325154,557.0,1835.0,NaN,exact Clopper-Pearson interval (statsmodels pr...
2,risk,Male,None,0.445188,0.426293,0.464202,1198.0,2691.0,NaN,exact Clopper-Pearson interval (statsmodels pr...
3,risk difference,Female,Male,-0.141645,-0.169570,-0.113213,NaN,NaN,1.055797e-21,"Newcombe's score interval (statsmodels, as R D..."
4,number needed for one event fewer,Female,Male,7.059882,5.897287,8.832898,NaN,NaN,NaN,1 / risk difference; its interval from the dif...
5,risk ratio,Female,Male,0.681830,0.628697,0.739454,NaN,NaN,1.055797e-21,Wald interval on the log scale (statsmodels Ta...
6,odds ratio,Female,Male,0.543159,0.479227,0.615621,NaN,NaN,1.055797e-21,Wald interval on the log scale (statsmodels Ta...


In [6]:
admitted = load("applicants")
admitted.keep_rows_where("admit", is_equal_to="Admitted")
events = {row["sex"]: row for row in read_rows(quiet(admitted.count_rows, for_each_group_in="sex"))}
sizes = {row["sex"]: row for row in read_rows(quiet(berkeley.count_rows, for_each_group_in="sex"))}
count_column = [c for c in next(iter(sizes.values())) if c != "sex"][0]
ds = "UCB admissions 1973 (Bickel et al. 1975)"


def risk_values(found):
    values = {}
    for row in found:
        for column in row:
            if not (column == "estimate" or (column.startswith("estimate ") and "CI" in column)):
                continue
            suffix = column[len("estimate"):]
            if row["measure"] == "risk":
                values[f"risk{suffix}, {row['group']}"] = row[column] if suffix else None
            elif row["measure"] in ("risk difference", "risk ratio", "odds ratio"):
                values[row["measure"] + suffix] = row[column]
    return {k: v for k, v in values.items() if v is not None}


def p_value_of(found, measure="risk difference"):
    return [row for row in found if row["measure"] == measure][0]["p-value"]


calls = {"table.compare_risks": lambda **o: rows(berkeley.compare_risks, "admit", for_each_group_in="sex", against="Male",
                                                  value="Admitted", **o),
         "sdata.compare_risks": lambda **o: rows(sdata.compare_risks,
                                                  events=[events["Female"][count_column], events["Male"][count_column]],
                                                  out_of=[sizes["Female"][count_column], sizes["Male"][count_column]],
                                                  names=["Female", "Male"], against="Male", **o)}
for word, call in calls.items():
    for interval in (95, 90):
        found = call(interval=interval)
        checks.check_values(ds, f"{word} (interval={interval})", risk_values(found))
        if interval == 95:
            checks.check(ds, word, "p-value", p_value_of(found))
    found = call(p_value_by="exact")
    checks.check(ds, f'{word} (p_value_by="exact")', "p-value", [row["p-value"] for row in found if row["p-value"] is not None][0])
    for options in [{"sides": 1, "higher": "Female"}, {"sides": 1, "lower": "Female"}, {"alternative": "less"}]:
        checks.check(ds, f"{word} (" + ", ".join(f"{k}={v!r}" for k, v in options.items()) + ")", "p-value",
                     p_value_of(call(**options)))
    found = [row for row in call(same_within=5) if "equivalence" in row["measure"]][0]
    checks.check_values(ds, f"{word} (same_within=)", found)
    for option in ("not_lower", "not_higher"):
        found = [row for row in call(same_within=5, **{option: "Female"}) if row["p-value"] is not None][-1]
        checks.check(ds, f'{word} (same_within=, {option}="Female")', "p-value", found["p-value"])

Kept 1755 of 4526 rows.


agrees  UCB admissions 1973 (Bickel et al. 1975) | table.compare_risks (interval=95) | risk 95 % CI low, Female: simplydata 0.282560451876, R binom.test 0.282560451876; relative difference 5.89e-16 (allowed 1e-10)
      Source: R binom.test (run reference) - https://search.r-project.org/R/refmans/stats/html/binom.test.html
agrees  UCB admissions 1973 (Bickel et al. 1975) | table.compare_risks (interval=95) | risk 95 % CI high, Female: simplydata 0.325153835485, R binom.test 0.325153835485; relative difference 0.00e+00 (allowed 1e-10)
      Source: R binom.test (run reference) - https://search.r-project.org/R/refmans/stats/html/binom.test.html
agrees  UCB admissions 1973 (Bickel et al. 1975) | table.compare_risks (interval=95) | risk 95 % CI low, Male: simplydata 0.426292720286, R binom.test 0.426292720286; relative difference 0.00e+00 (allowed 1e-10)
      Source: R binom.test (run reference) - https://search.r-project.org/R/refmans/stats/html/binom.test.html
agrees  UCB admissions 197

agrees  UCB admissions 1973 (Bickel et al. 1975) | table.compare_risks (p_value_by="exact") | p-value: simplydata 4.83590317934e-22, R fisher.test 4.83590317934e-22; relative difference 1.75e-14 (allowed 1e-10)


      Source: R fisher.test (run reference) - https://search.r-project.org/R/refmans/stats/html/fisher.test.html


agrees  UCB admissions 1973 (Bickel et al. 1975) | table.compare_risks (sides=1, higher='Female') | p-value: simplydata 1, R prop.test (Yates) 1; relative difference 0.00e+00 (allowed 1e-10)
      Source: R prop.test (Yates) (run reference) - https://search.r-project.org/R/refmans/stats/html/prop.test.html
agrees  UCB admissions 1973 (Bickel et al. 1975) | table.compare_risks (sides=1, lower='Female') | p-value: simplydata 5.27898404391e-22, R prop.test (Yates) 5.27898404391e-22; relative difference 0.00e+00 (allowed 1e-10)
      Source: R prop.test (Yates) (run reference) - https://search.r-project.org/R/refmans/stats/html/prop.test.html
agrees  UCB admissions 1973 (Bickel et al. 1975) | table.compare_risks (alternative='less') | p-value: simplydata 5.27898404391e-22, R prop.test (Yates) 5.27898404391e-22; relative difference 0.00e+00 (allowed 1e-10)
      Source: R prop.test (Yates) (run reference) - https://search.r-project.org/R/refmans/stats/html/prop.test.html


agrees  UCB admissions 1973 (Bickel et al. 1975) | table.compare_risks (same_within=) | p-value against the low margin: simplydata 0.999999999905, two one-sided Wald z tests, SciPy 0.999999999905; relative difference 0.00e+00 (allowed 1e-10)
      Source: two one-sided Wald z tests, SciPy (run reference)
agrees  UCB admissions 1973 (Bickel et al. 1975) | table.compare_risks (same_within=) | p-value against the high margin: simplydata 8.79411913396e-41, two one-sided Wald z tests, SciPy 8.79411913396e-41; relative difference 0.00e+00 (allowed 1e-10)
      Source: two one-sided Wald z tests, SciPy (run reference)
agrees  UCB admissions 1973 (Bickel et al. 1975) | table.compare_risks (same_within=, not_lower="Female") | p-value: simplydata 0.999999999905, one-sided Wald z test, SciPy 0.999999999905; relative difference 0.00e+00 (allowed 1e-10)
      Source: one-sided Wald z test, SciPy (run reference)
agrees  UCB admissions 1973 (Bickel et al. 1975) | table.compare_risks (same_within=, no

agrees  UCB admissions 1973 (Bickel et al. 1975) | sdata.compare_risks (p_value_by="exact") | p-value: simplydata 4.83590317934e-22, R fisher.test 4.83590317934e-22; relative difference 1.75e-14 (allowed 1e-10)
      Source: R fisher.test (run reference) - https://search.r-project.org/R/refmans/stats/html/fisher.test.html
agrees  UCB admissions 1973 (Bickel et al. 1975) | sdata.compare_risks (sides=1, higher='Female') | p-value: simplydata 1, R prop.test (Yates) 1; relative difference 0.00e+00 (allowed 1e-10)
      Source: R prop.test (Yates) (run reference) - https://search.r-project.org/R/refmans/stats/html/prop.test.html


agrees  UCB admissions 1973 (Bickel et al. 1975) | sdata.compare_risks (sides=1, lower='Female') | p-value: simplydata 5.27898404391e-22, R prop.test (Yates) 5.27898404391e-22; relative difference 0.00e+00 (allowed 1e-10)
      Source: R prop.test (Yates) (run reference) - https://search.r-project.org/R/refmans/stats/html/prop.test.html


agrees  UCB admissions 1973 (Bickel et al. 1975) | sdata.compare_risks (alternative='less') | p-value: simplydata 5.27898404391e-22, R prop.test (Yates) 5.27898404391e-22; relative difference 0.00e+00 (allowed 1e-10)
      Source: R prop.test (Yates) (run reference) - https://search.r-project.org/R/refmans/stats/html/prop.test.html
agrees  UCB admissions 1973 (Bickel et al. 1975) | sdata.compare_risks (same_within=) | p-value against the low margin: simplydata 0.999999999905, two one-sided Wald z tests, SciPy 0.999999999905; relative difference 0.00e+00 (allowed 1e-10)
      Source: two one-sided Wald z tests, SciPy (run reference)
agrees  UCB admissions 1973 (Bickel et al. 1975) | sdata.compare_risks (same_within=) | p-value against the high margin: simplydata 8.79411913396e-41, two one-sided Wald z tests, SciPy 8.79411913396e-41; relative difference 0.00e+00 (allowed 1e-10)
      Source: two one-sided Wald z tests, SciPy (run reference)
agrees  UCB admissions 1973 (Bickel et al. 1975

**Allowing for the department**: the Mantel-Haenszel odds ratio and risk ratio, the Cochran-Mantel-Haenszel test
and the Breslow-Day test.

In [7]:
found = {row["measure"]: row for row in rows(berkeley.compare_risks, "admit", for_each_group_in="sex", against="Male",
                                              value="Admitted", allowing_for="department")}
odds = [m for m in found if m.startswith("odds ratio allowing")][0]
risk = [m for m in found if m.startswith("risk ratio allowing")][0]
breslow = [m for m in found if "Breslow-Day" in m][0]
checks.check_values("UCB admissions by department (R mantelhaen.test example)", 'compare_risks (allowing_for="department")', {
    "Mantel-Haenszel odds ratio": found[odds]["estimate"],
    "Mantel-Haenszel odds ratio 95 % CI low": found[odds]["estimate 95 % CI low"],
    "Mantel-Haenszel odds ratio 95 % CI high": found[odds]["estimate 95 % CI high"],
    "Cochran-Mantel-Haenszel p-value": found[odds]["p-value"],
    "Mantel-Haenszel risk ratio": found[risk]["estimate"],
    "Mantel-Haenszel risk ratio 95 % CI low": found[risk]["estimate 95 % CI low"],
    "Mantel-Haenszel risk ratio 95 % CI high": found[risk]["estimate 95 % CI high"],
    "Breslow-Day p-value": found[breslow]["p-value"]})

agrees  UCB admissions by department (R mantelhaen.test example) | compare_risks (allowing_for="department") | Mantel-Haenszel odds ratio: simplydata 1.1053426615, R mantelhaen.test 1.1053426615; relative difference 0.00e+00 (allowed 1e-10)
      Source: R mantelhaen.test (run reference) - https://search.r-project.org/R/refmans/stats/html/mantelhaen.test.html
agrees  UCB admissions by department (R mantelhaen.test example) | compare_risks (allowing_for="department") | Mantel-Haenszel odds ratio 95 % CI low: simplydata 0.943102828516, R mantelhaen.test 0.943102828516; relative difference 2.35e-16 (allowed 1e-10)
      Source: R mantelhaen.test (run reference) - https://search.r-project.org/R/refmans/stats/html/mantelhaen.test.html
agrees  UCB admissions by department (R mantelhaen.test example) | compare_risks (allowing_for="department") | Mantel-Haenszel odds ratio 95 % CI high: simplydata 1.29549224368, R mantelhaen.test 1.29549224368; relative difference 1.71e-16 (allowed 1e-10)
    

8

## `compare_balanced_groups_of`: `against=` and `interval=` (lalonde)

The effect against the other group must be the same size with the other sign and the same standard error, and the
90 % interval the estimate plus and minus the right percentile times the standard error (the references were
worked out from simplydata's own 95 % answer in the reference script; the fits themselves are compared with
WeightIt, MatchIt and cobalt in notebook 16).

In [8]:
jobs = quiet(sdata.table, "../data/fetched/lalonde.csv", separator=",")
COLUMNS = ["age", "educ", "race", "married", "nodegree", "re74", "re75"]
ds = "lalonde (Dehejia & Wahba 1999)"
other = read_row(quiet(jobs.compare_balanced_groups_of, "re78", for_each_group_in="treat", allowing_for=COLUMNS,
                       test_method="weighting", against=1))
checks.check_values(ds, 'compare_balanced_groups_of (test_method="weighting", against=1)',
                    {"effect, the other way round": other["estimate"], "standard error": other["standard error"]})
for method, extra in [("weighting", {}), ("matching", {"effect_for": 1})]:
    ninety = read_row(quiet(jobs.compare_balanced_groups_of, "re78", for_each_group_in="treat", allowing_for=COLUMNS,
                            test_method=method, interval=90, **extra))
    checks.check_values(ds, f'compare_balanced_groups_of (test_method="{method}", interval=90)', ninety)

agrees  lalonde (Dehejia & Wahba 1999) | compare_balanced_groups_of (test_method="weighting", against=1) | effect, the other way round: simplydata -224.676308271, minus the effect against 0 -224.676308271; relative difference 7.21e-15 (allowed 1e-06)
      Source: minus the effect against 0 (run reference)
agrees  lalonde (Dehejia & Wahba 1999) | compare_balanced_groups_of (test_method="weighting", against=1) | standard error: simplydata 876.193185444, the same standard error 876.193185589; relative difference 1.66e-10 (allowed 1e-06)
      Source: the same standard error (run reference)


agrees  lalonde (Dehejia & Wahba 1999) | compare_balanced_groups_of (test_method="weighting", interval=90) | estimate 90 % CI low: simplydata -1216.53323096, estimate ± z(0.95) SE -1216.53323096; relative difference 1.87e-16 (allowed 1e-09)
      Source: estimate ± z(0.95) SE (run reference)
agrees  lalonde (Dehejia & Wahba 1999) | compare_balanced_groups_of (test_method="weighting", interval=90) | estimate 90 % CI high: simplydata 1665.8858475, estimate ± z(0.95) SE 1665.8858475; relative difference 5.46e-16 (allowed 1e-09)
      Source: estimate ± z(0.95) SE (run reference)
agrees  lalonde (Dehejia & Wahba 1999) | compare_balanced_groups_of (test_method="matching", interval=90) | estimate 90 % CI low: simplydata -268.559793398, estimate ± t(0.95, 368 df) SE -268.559793398; relative difference 2.54e-15 (allowed 1e-09)
      Source: estimate ± t(0.95, 368 df) SE (run reference)
agrees  lalonde (Dehejia & Wahba 1999) | compare_balanced_groups_of (test_method="matching", interval=90) | e

## Result

In [9]:
checks.finish()

22_risks_and_rates: 168 comparisons - 168 agrees
of the SKIPs, 0 are waiting for a simplydata word (see the reason column)


,notebook,dataset,method,quantity,simplydata,reference,reference source,difference,tolerance,tolerance type,score,status,reason,result,finding,explanation,link,kind,url,ran on
0,22_risks_and_rates,"lung cancer, Fredericia and other cities (R po...","table.compare_rates (per=1000, interval=95)","rate, Fredericia",13.750000,13.750000,events / time,0.000000e+00,1.000000e-10,relative,0.000000e+00,PASS,,agrees,,,,run reference,,"GitHub Actions, Linux 7.0.0-1020-azure, Intel(..."
1,22_risks_and_rates,"lung cancer, Fredericia and other cities (R po...","table.compare_rates (per=1000, interval=95)","rate 95 % CI low, Fredericia",6.863950,6.863950,"Garwood exact, SciPy",0.000000e+00,1.000000e-10,relative,0.000000e+00,PASS,,agrees,,,,run reference,,"GitHub Actions, Linux 7.0.0-1020-azure, Intel(..."
2,22_risks_and_rates,"lung cancer, Fredericia and other cities (R po...","table.compare_rates (per=1000, interval=95)","rate 95 % CI high, Fredericia",24.602548,24.602548,"Garwood exact, SciPy",0.000000e+00,1.000000e-10,relative,0.000000e+00,PASS,,agrees,,,,run reference,,"GitHub Actions, Linux 7.0.0-1020-azure, Intel(..."
3,22_risks_and_rates,"lung cancer, Fredericia and other cities (R po...","table.compare_rates (per=1000, interval=95)","rate, other cities",6.974427,6.974427,events / time,0.000000e+00,1.000000e-10,relative,0.000000e+00,PASS,,agrees,,,,run reference,,"GitHub Actions, Linux 7.0.0-1020-azure, Intel(..."
4,22_risks_and_rates,"lung cancer, Fredericia and other cities (R po...","table.compare_rates (per=1000, interval=95)","rate 95 % CI low, other cities",4.317280,4.317280,"Garwood exact, SciPy",0.000000e+00,1.000000e-10,relative,0.000000e+00,PASS,,agrees,,,,run reference,,"GitHub Actions, Linux 7.0.0-1020-azure, Intel(..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
163,22_risks_and_rates,lalonde (Dehejia & Wahba 1999),"compare_balanced_groups_of (test_method=""weigh...",standard error,876.193185,876.193186,the same standard error,-1.451494e-07,1.000000e-06,relative,1.656592e-10,PASS,,agrees,,,,run reference,,"GitHub Actions, Linux 7.0.0-1020-azure, Intel(..."
164,22_risks_and_rates,lalonde (Dehejia & Wahba 1999),"compare_balanced_groups_of (test_method=""weigh...",estimate 90 % CI low,-1216.533231,-1216.533231,estimate ± z(0.95) SE,-2.273737e-13,1.000000e-09,relative,1.869030e-16,PASS,,agrees,,,,run reference,,"GitHub Actions, Linux 7.0.0-1020-azure, Intel(..."
165,22_risks_and_rates,lalonde (Dehejia & Wahba 1999),"compare_balanced_groups_of (test_method=""weigh...",estimate 90 % CI high,1665.885847,1665.885847,estimate ± z(0.95) SE,-9.094947e-13,1.000000e-09,relative,5.459526e-16,PASS,,agrees,,,,run reference,,"GitHub Actions, Linux 7.0.0-1020-azure, Intel(..."
166,22_risks_and_rates,lalonde (Dehejia & Wahba 1999),"compare_balanced_groups_of (test_method=""match...",estimate 90 % CI low,-268.559793,-268.559793,"estimate ± t(0.95, 368 df) SE",-6.821210e-13,1.000000e-09,relative,2.539922e-15,PASS,,agrees,,,,run reference,,"GitHub Actions, Linux 7.0.0-1020-azure, Intel(..."
